***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 3 章：干涉测量中的位置天文学](3_0_introduction.ipynb)
    * 上一节：[3.2 时角、地方恒星时与过中天](3_2_hour_angle.ipynb)
    * 下一节：[3.4 方向余弦、相位中心与局部成像坐标](3_4_direction_cosine_coordinates.ipynb)

***


导入标准模块:


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


导入本节所需的专用模块:


In [ ]:
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
HTML('../style/code_toggle.html')


## 3.3 地平坐标、望远镜指向与可观测性<a id='pos:sec:horizontal'></a>

如果说赤道坐标回答的是“这个源在天球上被目录如何记录”，那么地平坐标回答的就是“望远镜此刻应该指向哪里”。对实际观测而言，这一步至关重要，因为望远镜的指向、最低仰角约束、地平遮挡以及低仰角大气路径长度，都必须在本地坐标系中讨论。


### 地平坐标的定义

地平坐标系以观测者所在地点的水平面为基面。我们记方位角为 `\mathcal{A}`，高度角（或仰角）为 `\mathcal{E}`。

* 方位角 `\mathcal{A}`：从正北方向沿地平圈向东量起；
* 高度角 `\mathcal{E}`：从地平线向上量起，地平线为 `0^\circ`，天顶为 `90^\circ`。

不同软件中方位角零点和正方向的约定可能略有不同，因此在把公式写成代码时必须检查约定。本书采用“从北向东”的常见天文约定。


<a id='pos:fig:horizontal'></a><img src='figures/horizontal.svg' width='40%'>

*图 3.3.1*：地平坐标系中的方位角 `\mathcal{A}` 与高度角 `\mathcal{E}`。


### 与赤道坐标之间的转换

给定台站纬度 `\varphi`、天源赤纬 `\delta` 和时角 `H` 后，地平坐标可由下式得到：

\[
\sin \mathcal{E} = \sin\varphi\,\sin\delta + \cos\varphi\,\cos\delta\,\cos H,
\]

\[
\cos \mathcal{E}\,\sin \mathcal{A} = -\cos\delta\,\sin H,
\qquad
\cos \mathcal{E}\,\cos \mathcal{A} = \sin\delta\,\cos\varphi - \cos\delta\,\sin\varphi\,\cos H.
\]

在程序中，通常用 `atan2` 同时处理 `\sin \mathcal{A}` 和 `\cos \mathcal{A}`，以保证方位角落在正确象限中。对后续分析而言，最重要的是第一式：它直接告诉我们源的仰角如何随 `H` 变化。


<a id='pos:fig:horizontal_triangle'></a><img src='figures/spher_trig.svg' width='42%'>

*图 3.3.2*：由天源、天顶和天极构成的球面三角。赤道坐标到地平坐标的转换可以由这个球面三角直接推得。


<div class='advice'>
<b>观测意义：</b> 当 `H = 0` 时，源过中天，此时高度角达到极值。若源在上中天经过南方或北方子午圈，其最大高度可写成
\[
\mathcal{E}_{\max} = 90^\circ - |\varphi - \delta|.
\]
这个量对观测计划极其重要：它决定了目标源是否能升到足够高的仰角，从而避免严重的大气吸收、较长的湿延迟路径和较差的增益稳定性。
</div>


### 为什么干涉仪尤其关心地平坐标

单天线和干涉阵都需要望远镜指向，但干涉阵对地平坐标更敏感，原因有三点。

* **可观测性窗口**：阵列常常规定最低工作仰角，例如 `15^\circ` 或 `20^\circ`；
* **大气与系统温度**：低仰角时有效气团厚度增大，透明度、相位稳定性和系统温度都可能变差；
* **阵列工程约束**：真实天线存在地平遮挡、机械限位、馈源绕线和扫描速度限制。

因此，地平坐标不是赤道坐标的附属量，而是把“天球几何”变成“可执行观测”的关键一步。


### 示例 1：MeerKAT 台址上不同赤纬源的仰角曲线

下面用 MeerKAT 附近纬度 `\varphi \approx -30.7^\circ` 做一个纯几何例子，直接画出不同赤纬天源随时角变化的仰角。这样可以一眼看出哪些源会高高升起、哪些源只是擦着地平线掠过，以及哪些源根本无法被该台站观测到。


In [ ]:
lat_deg = -30.713
hour_angle_h = np.linspace(-12, 12, 1201)
declinations = np.array([-80, -60, -30, 0, 30])


def equatorial_to_horizontal(dec_deg, hour_angle_h, lat_deg):
    H = np.deg2rad(15.0 * hour_angle_h)
    dec = np.deg2rad(dec_deg)
    lat = np.deg2rad(lat_deg)

    sin_el = np.sin(lat) * np.sin(dec) + np.cos(lat) * np.cos(dec) * np.cos(H)
    el = np.arcsin(np.clip(sin_el, -1.0, 1.0))

    sin_az = -np.cos(dec) * np.sin(H) / np.cos(el)
    cos_az = (np.sin(dec) - np.sin(lat) * np.sin(el)) / (np.cos(lat) * np.cos(el))
    az = (np.rad2deg(np.arctan2(sin_az, cos_az)) + 360.0) % 360.0
    return az, np.rad2deg(el)


fig, ax = plt.subplots(figsize=(10, 5.5))
for dec in declinations:
    _, elevation = equatorial_to_horizontal(dec, hour_angle_h, lat_deg)
    ax.plot(hour_angle_h, elevation, lw=2, label=fr'$\delta={dec:+d}^\circ$')

ax.axhline(0, color='0.25', ls=':')
ax.axhline(20, color='tab:red', ls='--', alpha=0.7, label='20 deg elevation limit')
ax.set_xlim(-12, 12)
ax.set_ylim(-20, 90)
ax.set_xlabel('Hour angle H [h]')
ax.set_ylabel('Elevation [deg]')
ax.set_title('MeerKAT: elevation tracks for several declinations')
ax.grid(alpha=0.3)
ax.legend(ncol=2)
plt.show()


从这张图里可以读出几条非常实用的规则。位于台站纬度附近的天源会在过中天时达到最高仰角；较南的源可能成为拱极源，全天都在地平线以上；而赤纬过高的北天源则可能根本无法升起。对南非台址来说，`\delta = +30^\circ` 的源已经明显很低，而更北的源会直接消失在可观测范围之外。


### 示例 2：最低仰角约束如何改变可观测时间窗

很多阵列并不会在地平线 `0^\circ` 就开始观测，而是要求目标源至少高于某个安全阈值，例如 `20^\circ`。下面把这个条件直接翻译成“可连续跟踪多少小时”，看看它如何随赤纬变化。


In [ ]:
lat_deg = -30.713
hour_angle_full_h = np.linspace(-12, 12, 2401)
dec_grid = np.linspace(-90, 90, 361)
min_elevation_deg = 20.0
visible_hours = []

for dec in dec_grid:
    _, elevation = equatorial_to_horizontal(dec, hour_angle_full_h, lat_deg)
    visible_hours.append(np.mean(elevation >= min_elevation_deg) * 24.0)

visible_hours = np.array(visible_hours)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dec_grid, visible_hours, color='tab:green', lw=2)
ax.axvline(lat_deg, color='tab:purple', ls='--', label=fr'Site latitude $\varphi={lat_deg:.1f}^\circ$')
ax.set_xlim(-90, 90)
ax.set_ylim(0, 24)
ax.set_xlabel('Source declination [deg]')
ax.set_ylabel('Visible time above 20 deg [sidereal h]')
ax.set_title('MeerKAT: observability window vs. declination')
ax.grid(alpha=0.3)
ax.legend()
plt.show()


这张图已经非常接近观测计划中的真实问题了：给定目标源赤纬和台站纬度，我们可以快速估计它在满足最低仰角约束下能跟踪多久。对于需要长时间自转合成的观测，这个时间窗直接决定了 `uv` 覆盖的完整程度；对于低仰角敏感的高频观测，它甚至会决定某个源是否值得在该台站上安排。


### 示例 3：从目录位置直接得到台站指向轨迹

前面三节的链条其实已经可以连起来了。下面以 3C286 为例，把目录中的赤经赤纬直接转换成 “给定地方恒星时时，VLA 该把天线指向哪里”。这个例子把第 3.1 节的目录位置、第 3.2 节的时角定义和本节的地平坐标放进了同一段代码里。


In [ ]:
source_name = '3C286'
source_ra_h = 13 + 31 / 60.0 + 8.3 / 3600.0
source_dec_deg = 30 + 30 / 60.0 + 33.0 / 3600.0
lat_vla_deg = 34.08
lst_h = np.linspace(0, 24, 721)
hour_angle_h = ((lst_h - source_ra_h + 12.0) % 24.0) - 12.0
azimuth_deg, elevation_deg = equatorial_to_horizontal(source_dec_deg, hour_angle_h, lat_vla_deg)
visible = elevation_deg > 0.0

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(lst_h, elevation_deg, lw=2, color='tab:blue')
ax1.axvline(source_ra_h, color='tab:red', ls='--', label='LST = RA (transit)')
ax1.axhline(20.0, color='tab:orange', ls=':', label='20 deg elevation limit')
ax1.set_xlim(0, 24)
ax1.set_ylim(-20, 90)
ax1.set_xlabel('Local sidereal time LST [h]')
ax1.set_ylabel('Elevation [deg]')
ax1.set_title(f'{source_name} at the VLA: elevation curve')
ax1.grid(alpha=0.3)
ax1.legend(fontsize=9)

ax2 = fig.add_subplot(1, 2, 2, projection='polar')
ax2.set_theta_zero_location('N')
ax2.set_theta_direction(-1)
ax2.plot(np.deg2rad(azimuth_deg[visible]), 90.0 - elevation_deg[visible], lw=2, color='tab:green')
ax2.set_rlim(90, 0)
ax2.set_title(f'{source_name} in the local sky')

plt.tight_layout()
plt.show()


这段代码的意义，在于它把第 3 章前半部分的所有核心变量都串了起来：源目录给出 `RA/Dec`，地方恒星时给出 `H`，而 `H` 与台站纬度一起决定本地天空中的方位角和高度角。到这一步为止，观测调度已经具备了最基本的几何输入。


#### 本节要点

* 地平坐标把天球上的目录位置转化为台站本地的指向位置，是观测执行层面的核心坐标；
* 由 `H`、`\delta` 和 `\varphi` 可以直接求得方位角与高度角，其中仰角公式最常用于可观测性分析；
* 最低仰角阈值、大气路径长度和工程限位，使得“理论上可见”与“适合观测”并不是同一个概念；
* 一旦地平坐标已知，我们就能回答望远镜要如何指向；接下来还需要一个围绕相位中心定义的局部像面坐标，这就是方向余弦 `(l,m,n)`。


***

下一节：[3.4 方向余弦、相位中心与局部成像坐标](3_4_direction_cosine_coordinates.ipynb)
